# Viora — Free GPU Training (Colab / Kaggle)

Train Viora's **own** video-language model for **$0** on a free **T4 (16 GB)** — plenty for the
SigLIP + Qwen-0.5B + LoRA ("pragmatic") model. Free sessions time out (4–12 h), so this notebook
**checkpoints** to Drive/output and can **resume** across sessions.

**Before running:** enable the GPU.
- **Colab:** Runtime → Change runtime type → **T4 GPU**
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** (30 free GPU-hrs/week)

This is Viora's own model — no wrapper, no external answering API.

In [ ]:
# 1) Confirm the free GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Get the Viora code.
#    Easiest: push this repo to GitHub, then set REPO_URL below.
#    (locally:  git init && git add -A && git commit -m init && git remote add origin <url> && git push -u origin main)
REPO_URL = "https://github.com/YOUR_USERNAME/viora.git"  # <-- set this
import os
if not os.path.isdir('viora'):
    !git clone $REPO_URL viora
%cd viora
# Alternative if you don't want GitHub: zip the repo, upload it here, then:  !unzip -q viora.zip

In [ ]:
# 3) Install Viora + training deps (Colab/Kaggle already ship a CUDA-enabled torch)
!pip install -q -e ".[all]" peft webdataset
!python scripts/validate_environment.py

In [ ]:
# 4) Choose an output dir that SURVIVES session end (so you can resume).
#    Colab -> Google Drive;  Kaggle -> /kaggle/working (downloadable).
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/viora_runs/pragmatic'
except Exception:
    OUT = '/kaggle/working/pragmatic' if os.path.isdir('/kaggle') else 'runs/pragmatic'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

In [ ]:
# 5) Data. Start with SYNTHETIC shards so the whole pipeline runs end-to-end for free.
#    Swap in a real dataset (e.g. MSR-VTT) later for a useful model — see docs/PRODUCTION.md.
!python scripts/build_shards.py --synthetic 3000 --out data/shards/train-%06d.tar
SHARDS = 'data/shards/train-{000000..000002}.tar'

In [ ]:
# 6) Train (LoRA on Qwen-0.5B + SigLIP + Viora's bridge). Checkpoints every 200 steps.
!python scripts/train.py \
  --model configs/model/viora_pragmatic.yaml \
  --train configs/training/pragmatic_lora.yaml \
  --shards "{SHARDS}" \
  llm.name_or_path=Qwen/Qwen2.5-0.5B-Instruct \
  training.precision=bf16 training.batch_size=4 training.gradient_checkpointing=true \
  training.max_steps=3000 training.save_every=200 training.log_every=20 \
  training.output_dir={OUT}

## Resuming after a session times out

Re-run cells 1–5, then add `training.resume=<checkpoint>` to cell 6 — e.g.:

```
  training.resume={OUT}/step_2000.pt
```

It restores model + optimizer + step and continues. Repeat across free sessions until done.

## Get real answers

Synthetic data proves the loop; for a *useful* model, upload a real video-QA set (MSR-VTT ~7 GB)
as a Kaggle Dataset / to Drive, convert with `scripts/build_shards.py`, and point `--shards` at it.
Bigger LLM (`Qwen/Qwen2.5-1.5B-Instruct`) + more data + more hours = better quality.

## Serve it

```
VIORA_MODEL_CONFIG=configs/model/viora_pragmatic.yaml VIORA_CHECKPOINT={OUT}/final.pt \
  uvicorn viora.serving.api:app --host 0.0.0.0 --port 8000
```